# 1D CNN-A — Reconstruction Quality

Loads the trained model, runs a sample of windows through the full
encoder → decoder path, and compares input vs. output side by side.
Answers the key question: did the autoencoder learn genuine market
structure, or did it find a trivial shortcut?

> **Prerequisite:** run `1dcnn_train.ipynb` first — it saves `model.pt` to
> `DATA_DIR / SYMBOL /`.

## 1. Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from config import Config
from data import (
    load_bars, clean_data, add_features, drop_feature_nans,
    scale_features, make_windows, filter_gap_windows,
)
from model import ConvAutoencoder, WindowDataset, load_model

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

In [ ]:
from config import Config

cfg = Config()

# ── Override defaults here before running the rest of the notebook ────────────
# cfg.MAX_BARS       = None   # load all bars (~552k)
# cfg.EPOCHS         = 30     # full training run
# cfg.N_CLUSTERS     = 12     # try more/fewer clusters
# cfg.LATENT_DIM     = 64     # larger latent space
# cfg.N_SAMPLE       = 5_000  # render more windows in Section 9

# Expose all config fields as module-level names so every downstream cell
# can use SYMBOL, WINDOW_SIZE, LR, feature_cols, DEVICE, etc. unchanged.
globals().update(vars(cfg))

print(f"Symbol={SYMBOL}  Timeframe={TIMEFRAME}  {START_DATE} → {END_DATE}")
print("Using device:", DEVICE)

## 3. Fetch Data from Alpaca API
Calls the local `alpaca_api` FastAPI service (must be running: `uv run main.py`).
Fetches TSLA 1-minute bars and saves to both DB and CSV.

In [ ]:
if FETCH_DATA:
    import httpx  # only needed when FETCH_DATA = True
    params = {
        "symbols": SYMBOL,
        "timeframe": TIMEFRAME,
        "start": START_DATE,
        "end": END_DATE,
        "save_to": "db,csv",
    }
    with httpx.Client(timeout=None) as client:
        r = client.get(f"{API_BASE}/bars", params=params)
        r.raise_for_status()
        result = r.json()
    bars = result.get("data", {}).get("bars", {}).get(SYMBOL, [])
    print(f"Fetched {len(bars)} bars for {SYMBOL}")
    print("Saved:", result.get("saved"))
else:
    print("FETCH_DATA=False — skipping. Set True in Config to re-pull.")

## 4. Load Data

In [ ]:
# Load raw OHLCV bars from the CSV file.
df = load_bars(DATA_DIR, SYMBOL, TIMEFRAME, MAX_BARS)

Check for:

Duplicate timestamps
Missing timestamps (gaps)
NaNs
Infinite values
Bad OHLC relationships (high < low, etc.)

Typical checks:

In [ ]:
# Remove duplicate timestamps and rows with missing values.
df = clean_data(df)

## 5. Verify Time Continuity
A CNN assumes a consistent sequence.

Look for:

missing bars
duplicate bars
irregular spacing

If you're using 1-minute candles, every row should be exactly 1 minutes apart.

In [ ]:
delta = df["timestamp"].diff()
dt = pd.to_timedelta(delta).dt.total_seconds()
print("Average time delta (seconds):", dt.mean())
print("Time delta distribution (seconds):")
print(dt.describe())

## 6. Add Features

In [ ]:
# Calculate 14 technical indicator columns (EMAs, MACD, candle shape, returns, volume ratio).
df = add_features(df)

## 6. Remove Initial NaNs

Feature engineering creates NaNs.

In [ ]:
# Drop the warm-up rows where EMAs and rolling means don't have enough history yet.
df = drop_feature_nans(df)

## 7. Scale Features

This is critical.

CNNs train poorly on:

close = 45000
volume = 10000000
return = 0.001

all mixed together.

StandardScaler

Most common:

In [ ]:
# Normalise every feature column so they all sit in a similar numeric range.
# RobustScaler uses the median and IQR — better than mean/std for financial data with outliers.
df, scaler = scale_features(df, feature_cols)

## 8. Create Fixed-Length Windows

A CNN does not ingest an entire dataframe.

It ingests samples.

In [ ]:
# Slice the time series into overlapping WINDOW_SIZE-bar windows.
# Each window is one training sample for the CNN.
X_raw = make_windows(df, feature_cols, WINDOW_SIZE)
n_features = len(feature_cols)

## 11. Filter Gap Windows
A window that spans an overnight or weekend gap mixes pre-gap and post-gap bars — the CNN would learn noise, not patterns. Any window whose 64-bar span crosses a gap > 5 minutes is dropped.

In [ ]:
# Remove windows that span overnight or weekend gaps.
# Such windows would teach the model noise rather than real patterns.
X_clean, valid_mask = filter_gap_windows(X_raw, df, WINDOW_SIZE)

## 13. Autoencoder Model
Encoder compresses `(batch, 14, 64)` → latent vector `(batch, LATENT_DIM)`.
Decoder reconstructs `(batch, 14, 64)` from the latent vector.
Training loss is reconstruction MSE — no labels needed.

In [ ]:
# ConvAutoencoder is defined in model.py — import it at the top.
# Here we just create an instance and move it to the device (CPU or GPU).
model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

In [ ]:
# WindowDataset is defined in model.py and imported at the top.
# It wraps a tensor so PyTorch's DataLoader can iterate it in batches.
# (No code needed here — the import at the top handles it.)

In [ ]:
# Load the weights saved by 1dcnn_train.ipynb.
# load_model() builds an empty ConvAutoencoder and fills it with the saved weights.
model = load_model(DATA_DIR, SYMBOL, n_features, LATENT_DIM, DEVICE)

## 18. Sample Windows for Reconstruction

In [ ]:
# Pick a random sample of windows to reconstruct
np.random.seed(42)
N_RECON = min(200, len(X_clean))
idx = np.random.choice(len(X_clean), size=N_RECON, replace=False)
sample_np = X_clean[idx]                         # (N_RECON, 64, 14)
sample_t  = torch.tensor(sample_np).permute(0, 2, 1).to(DEVICE)  # (N, 14, 64)

with torch.no_grad():
    recon_t = model(sample_t)                    # (N, 14, 64)

orig  = sample_t.cpu().numpy().transpose(0, 2, 1)   # (N, 64, 14)
recon = recon_t.cpu().numpy().transpose(0, 2, 1)     # (N, 64, 14)

mse_per_window = ((orig - recon) ** 2).mean(axis=(1, 2))  # (N,)
print(f"Reconstruction MSE  mean={mse_per_window.mean():.5f}  "
      f"min={mse_per_window.min():.5f}  max={mse_per_window.max():.5f}")

## 19. Side-by-Side: Original vs. Reconstructed

Each row shows one window. Blue = original, orange = reconstructed.
Pick 4 features to keep the chart readable — use `PLOT_FEATURES` from config.

In [ ]:
N_SHOW = 6   # number of windows to plot
plot_idx = [feature_cols.index(f) for f in PLOT_FEATURES]

fig, axes = plt.subplots(N_SHOW, len(PLOT_FEATURES), figsize=(18, N_SHOW * 2.2))
for row in range(N_SHOW):
    for col, (fi, fname) in enumerate(zip(plot_idx, PLOT_FEATURES)):
        ax = axes[row, col]
        ax.plot(orig[row, :, fi],  color='royalblue', lw=1.5, label='original')
        ax.plot(recon[row, :, fi], color='tomato',    lw=1.5, label='reconstructed', ls='--')
        if row == 0: ax.set_title(fname, fontsize=11)
        if col == 0: ax.set_ylabel(f'window {row}', fontsize=9)
        ax.tick_params(labelsize=7)
        if row == 0 and col == 0: ax.legend(fontsize=8)

plt.suptitle(f'Original vs. Reconstructed — {N_SHOW} random windows', fontsize=13)
plt.tight_layout()
plt.show()

## 20. Per-Feature Reconstruction Error

Which features does the autoencoder reconstruct accurately, and which does it struggle with?
A low MSE on a feature means the encoder captured its structure well.

In [ ]:
# MSE per feature across all N_RECON windows
feat_mse = ((orig - recon) ** 2).mean(axis=(0, 1))  # (14,)  mean over windows & time

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(range(len(feature_cols)), feat_mse,
              color=['tomato' if m > feat_mse.mean() else 'royalblue' for m in feat_mse])
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=10)
ax.axhline(feat_mse.mean(), color='gray', ls='--', lw=1, label=f'mean = {feat_mse.mean():.4f}')
ax.set_ylabel('Mean Squared Error')
ax.set_title('Per-Feature Reconstruction Error (red = above average)')
ax.legend()
plt.tight_layout()
plt.show()

print('Worst features:', sorted(zip(feature_cols, feat_mse), key=lambda x: -x[1])[:5])

## 21. Distribution of Reconstruction Errors

The histogram shows how reconstruction quality is spread across all sampled windows.
A tight peak near zero = consistent quality. A long right tail = some windows
the model struggles with — these are likely unusual market events or data anomalies.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(mse_per_window, bins=50, color='royalblue', edgecolor='white', lw=0.4)
ax.axvline(mse_per_window.mean(),   color='orange', ls='--', lw=1.5,
           label=f'mean = {mse_per_window.mean():.4f}')
ax.axvline(np.percentile(mse_per_window, 95), color='tomato', ls='--', lw=1.5,
           label=f'p95  = {np.percentile(mse_per_window, 95):.4f}')
ax.set_xlabel('Reconstruction MSE (per window)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Reconstruction Errors')
ax.legend()
plt.tight_layout()
plt.show()

n_outliers = (mse_per_window > np.percentile(mse_per_window, 95)).sum()
print(f'{n_outliers} windows in the top 5% of reconstruction error '
      f'(hardest for the model to reconstruct)')

## 22. What You're Seeing

### Side-by-Side Plots
If the orange dashed line tracks the blue line closely, the autoencoder faithfully
captured that feature's dynamics. If they diverge, the encoder compressed away
information the decoder can't recover.

**Good signs:** Close lines on `close` and `ema_9` (price trend preserved).  
**Normal:** `macd_hist` and `volume_ratio` may be noisier — they carry high-frequency
variation that a 32-dim bottleneck naturally smooths out.

### Per-Feature Error
Features the model reconstructs *poorly* are ones the latent space has compressed
away. That compression is intentional — those features add noise, not signal, from
the model's perspective.

### Error Distribution
A narrow distribution = the model treats most windows similarly (predictable market).  
A wide, heavy-tailed distribution = some windows are genuinely unusual — these are
worth inspecting as potential regime shifts or data quality issues.